# Générateurs

## Pipeline paresseux sur un gros fichier

Les générateurs enchaînent des transformations sans jamais stocker les données intermédiaires. La cellule suivante crée un journal de 500 000 requêtes (`numéro;utilisateur;durée_ms`) et fournit `peak_memory`, qui mesure le pic de mémoire d'un appel.

1. Écrivez trois générateurs&nbsp;: `read_lines(path)` qui produit les lignes du fichier, `parse(lines)` qui produit des tuples `(utilisateur, durée)`, et `slow(records, threshold)` qui ne garde que les requêtes plus longues que `threshold`.
2. Écrivez `total_slow_generators(path)`, qui enchaîne vos générateurs et renvoie avec `sum` la durée totale des requêtes de plus de 900 ms.
3. Écrivez `total_slow_lists(path)`, qui calcule la même chose en stockant chaque étape dans une liste (`readlines`, `list(...)`).
4. Comparez les résultats et les pics de mémoire des deux versions avec `peak_memory`. Les totaux doivent être égaux, et la version à base de listes doit utiliser des dizaines de mégaoctets de plus.

In [ ]:
import random
import tracemalloc

with open("requests.log", "w", encoding="utf8") as fh:
    for i in range(500_000):
        fh.write(f"{i};user{random.randrange(1_000)};{random.randrange(1_000)}\n")


def peak_memory(function):
    """Exécute function() et renvoie son résultat et son pic de mémoire en Mo."""
    tracemalloc.start()
    result = function()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return result, peak / 1e6

In [ ]:
# Votre code ici

### Solution

In [ ]:
from collections.abc import Iterable, Iterator


def read_lines(path: str) -> Iterator[str]:
    with open(path, encoding="utf8") as fh:
        yield from fh


def parse(lines: Iterable[str]) -> Iterator[tuple[str, int]]:
    for line in lines:
        _, user, duration = line.rstrip("\n").split(";")
        yield user, int(duration)


def slow(
    records: Iterable[tuple[str, int]], threshold: int
) -> Iterator[tuple[str, int]]:
    for user, duration in records:
        if duration > threshold:
            yield user, duration


def total_slow_generators(path: str) -> int:
    return sum(duration for _, duration in slow(parse(read_lines(path)), 900))


def total_slow_lists(path: str) -> int:
    with open(path, encoding="utf8") as fh:
        lines = fh.readlines()
    records = list(parse(lines))
    return sum(duration for _, duration in list(slow(records, 900)))


total_g, peak_g = peak_memory(lambda: total_slow_generators("requests.log"))
total_l, peak_l = peak_memory(lambda: total_slow_lists("requests.log"))
print(total_g == total_l, f"générateurs : {peak_g:.1f} Mo, listes : {peak_l:.1f} Mo")

Les générateurs ne gardent en mémoire qu'une ligne à la fois, quelle que soit la taille du fichier&nbsp;: la version à base de listes stocke tout le fichier, puis tous les tuples.

## Générateur de dates infini

Écrivez la fonction `gen_dates` qui, à partir d'une date donnée, génère toutes les dates successives séparées d'une semaine.

In [ ]:
import datetime
from collections.abc import Iterator


def gen_dates(date: datetime.date) -> Iterator[datetime.date]:
  pass  # Votre code ici

Proposez une méthode directe pour utiliser ce générateur infini.

Proposez aussi une méthode `take[T](iterable: Iterable[T], n: int) -> Iterator[T]` (qui prend en entrée un itérable et renvoie ses `n` premiers éléments) pour utiliser ce générateur infini.

In [ ]:
# Votre code ici

### Solution

In [ ]:
import datetime
from collections.abc import Iterable, Iterator


def gen_dates(date: datetime.date) -> Iterator[datetime.date]:
  week = datetime.timedelta(days=7)
  while True:
    date += week
    yield date


# Avec l'opérateur walrus
def gen_dates(date: datetime.date) -> Iterator[datetime.date]:
  week = datetime.timedelta(days=7)
  while True:
    yield (date := date + week)


for i, d in enumerate(gen_dates(datetime.date(1900, 1, 1))):
  if i == 10:
    break
  print(d)


def take[T](iterable: Iterable[T], n: int) -> Iterator[T]:
  for i, item in enumerate(iterable):
    if i == n:
      break
    yield item


def take[T](iterable: Iterable[T], n: int) -> Iterator[T]:
  for item, _ in zip(iterable, range(n)):
    yield item


for date in take(gen_dates(datetime.date(1900, 1, 1)), 10):
  print(date)

## Libérer une ressource avec `close` (difficile)

Un générateur peut détenir une ressource (fichier, connexion…) pendant qu'il produit ses valeurs.

Écrivez un générateur `numbered_lines(path)` qui affiche `ouverture`, ouvre le fichier et produit ses lignes numérotées `(numéro, ligne)` avec `yield from`, puis affiche `fermeture` quand le fichier est refermé, y compris si l'appelant s'arrête avant la fin.

Lisez les trois premières lignes de `requests.log` avec `next`, puis appelez `close()` sur le générateur&nbsp;: `fermeture` doit s'afficher immédiatement. Pourquoi&nbsp;?

In [ ]:
# Votre code ici

### Solution

In [ ]:
from collections.abc import Iterator


def numbered_lines(path: str) -> Iterator[tuple[int, str]]:
    print("ouverture")
    with open(path, encoding="utf8") as fh:
        try:
            yield from enumerate(fh, start=1)
        finally:
            print("fermeture")


lines = numbered_lines("requests.log")
for _ in range(3):
    print(next(lines))
lines.close()

`close()` lève `GeneratorExit` au point de pause du générateur&nbsp;: le bloc `finally` s'exécute et le `with` referme le fichier. Sans `close()`, le fichier ne serait refermé qu'à la destruction du générateur.

## Utilisation de la méthode `send` d'un générateur (très difficile)

Concevez un générateur qui, étant donné une somme de départ d'argent, va recevoir de votre part un taux d'intérêt et vous rendre la nouvelle valeur de la somme après application du taux.

In [ ]:
# Votre code ici

### Solution

In [ ]:
from collections.abc import Generator


def interests(amount: float) -> Generator[float, float, None]:
  while True:
    rate = yield amount
    amount *= rate
    # Ou directement
    # amount *= yield amount


amounts = interests(100)
print(next(amounts))  # Initialisation du générateur
for rate in [1.05, 1.08, 0.98]:
  print(amounts.send(rate))